# Fine-tune Dia TTS on Conversational Data

In this notebook, we fine-tune [Dia 1.6B](https://huggingface.co/nari-labs/Dia-1.6B-0626) on the [DailyTalk](https://huggingface.co/datasets/eustlb/dailytalk-conversations-grouped) conversational dataset for multi-speaker text-to-speech.

Dia is a 1.6B parameter TTS model from Nari Labs that can generate realistic dialogue with multiple speakers. Fine-tuning on conversational data improves its ability to produce natural-sounding multi-turn speech.

**What we'll cover:**
- Loading and preprocessing a conversational TTS dataset
- Formatting multi-speaker text with speaker tags
- Fine-tuning Dia with 8-bit AdamW
- Generating speech from the fine-tuned model

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes soundfile librosa

## Load Dataset

The DailyTalk dataset contains grouped multi-turn conversations with speaker IDs and corresponding audio. We resample to 44.1kHz as required by Dia.

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("eustlb/dailytalk-conversations-grouped", split="train[:100]")
dataset = dataset.cast_column("audio", Audio(sampling_rate=44_100))

print(f"Training samples: {len(dataset)}")
print(f"Features: {dataset.features}")
print(f"\nExample speaker IDs: {dataset[0]['speaker_ids']}")
print(f"Example texts: {dataset[0]['texts'][:2]}")

## Load Model and Processor

In [ ]:
import torch
from transformers import AutoProcessor, DiaForConditionalGeneration

model_checkpoint = "nari-labs/Dia-1.6B-0626"

processor = AutoProcessor.from_pretrained(model_checkpoint)
model = DiaForConditionalGeneration.from_pretrained(model_checkpoint)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model: {model_checkpoint}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Data Collator

Each sample contains multiple turns with speaker IDs. The collator formats them with speaker tags like `[S1]` and `[S2]`, then processes both text and audio through the Dia processor. Labels are masked at padding positions.

In [ ]:
class DiaDataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.pad_id = processor.tokenizer.pad_token_id

    def __call__(self, features):
        audios = []
        texts = []
        for f in features:
            grouped_text = ""
            for id, text in zip(f["speaker_ids"], f["texts"]):
                grouped_text += f"[S{id + 1}] {text} "
            texts.append(grouped_text)
            audios.append(f["audio"]["array"])

        proc_out = self.processor(
            audio=audios,
            text=texts,
            generation=False,
            output_labels=True,
            padding=True,
            return_tensors="pt",
        )
        proc_out["labels"][proc_out["labels"] == self.pad_id] = -100
        return proc_out

data_collator = DiaDataCollator(processor)

## Fine-tuning

We use 8-bit AdamW (`adamw_bnb_8bit`) to reduce optimizer memory, along with a cosine learning rate schedule and gradient accumulation.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./dia-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    max_steps=10,  # increase for real training
    bf16=True,
    logging_steps=5,
    save_steps=10,
    remove_unused_columns=False,
    report_to="none",
    optim="adamw_bnb_8bit",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

trainer.train()

## Save Model

In [ ]:
trainer.save_model("./dia-finetuned")
processor.save_pretrained("./dia-finetuned")
print("Model saved!")

## Inference

Generate speech from a multi-speaker dialogue prompt.

In [ ]:
import soundfile as sf

model.eval()

text = "[S1] Hey, how's it going? [S2] Pretty good, thanks for asking!"

inputs = processor(text=text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs)

audio_array = processor.decode(output)
sf.write("output.wav", audio_array, samplerate=44_100)
print("Generated audio saved to output.wav")